# ACL Python 算子调用：SpMV

按顺序运行下面的代码，需要修改的参数已在代码中标出。


<a id="s-05-03_spmv"></a>


## 5.3 SpMV：`acl.op.execute_v2`

### 小节概述

目标：先用 ATC 编译 `SparseTensorDenseMatMul` 单算子 OM，再通过 `acl.op.execute_v2` 提交 COO SpMV，并用 COO 累加结果校验。

<img src="images/spmv_gemm_data_contract.svg" width="760" style="display:block; margin-left:0;" />

<table style="text-align:left; margin-left:0;">
<tr><th>输入</th><th>格式要求</th><th>失败条件</th></tr>
<tr><td>indices</td><td><code>[NNZ,2]</code>，int64，依次为 row/col</td><td>越界、shape 错误；本实验还禁止重复坐标</td></tr>
<tr><td>values</td><td><code>[NNZ]</code>，float32</td><td>长度与 indices 不一致</td></tr>
<tr><td>dense_shape</td><td>shape=<code>[2]</code>，内容=<code>[M,N]</code>，int64</td><td>与 x/output 不一致</td></tr>
<tr><td>x2 / output</td><td><code>[N,1]</code> / <code>[M,1]</code>，float32</td><td>维度或 dtype 不匹配</td></tr>
</table>

> 本节在 `Ascend910_9362` / CANN 9.0 环境运行，将该算子标记为 AI CPU/tf_kernel 能力路径，`aicore_accelerated=false`。不能把算子在 NPU 系统中执行写成 AI Core 加速。


### COO 怎样表示矩阵

对矩阵 `[[2,0,3],[0,4,0]]`，COO 的 indices 为 `[[0,0],[0,2],[1,1]]`，values 为 `[2,3,4]`。每个坐标对应一个非零元。`dense_shape=[2,3]` 保存矩阵行列数，它本身是长度为 2 的一维数组。

接口将向量看作只有一列的矩阵，因此 x 的 shape 为 `[N,1]`，输出为 `[M,1]`。


In [ ]:
import json
import subprocess
import time
from pathlib import Path

import acl
import numpy as np

%run src/acl_helpers.py

SOC_VERSION = acl.get_soc_name()

# 学生修改：矩阵大小、非零元数量、设备编号和重复次数。
M, N, NNZ = 128, 128, 256
DEVICE_ID = 0
REPEAT = 10
MODEL_DIR = (Path("work") / "05.03_spmv" / "models").resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"M={M}, N={N}, NNZ={NNZ}, values/x/y=float32, device={DEVICE_ID}, repeat={REPEAT}, soc={SOC_VERSION}")


### 准备输入

修改并运行上面的参数单元格后，从这里向下重新运行。输入形状和单算子描述会使用同一组参数。


In [ ]:
linear = np.arange(NNZ, dtype=np.int64) * (M * N // NNZ)
indices = np.stack([linear // N, linear % N], axis=1)
values = ((np.arange(NNZ) * 7 + 1) % 13 + 1).astype(np.float32)
dense_shape = np.array([M, N], dtype=np.int64)
x2 = ((np.arange(N) * 3 + 1) % 11).astype(np.float32).reshape(N, 1)

assert indices.shape == (NNZ, 2) and values.shape == (NNZ,)
assert dense_shape.tolist() == [M, N] and x2.shape == (N, 1)
assert np.all((0 <= indices[:, 0]) & (indices[:, 0] < M))
assert np.all((0 <= indices[:, 1]) & (indices[:, 1] < N))
assert np.unique(indices, axis=0).shape[0] == NNZ, "本实验要求 COO 坐标唯一"


### 生成并编译单算子描述

ATC 根据下面的输入、输出和属性描述生成单算子 OM；输入顺序必须与 `SparseTensorDenseMatMul` 的参数要求一致。


In [ ]:
spmv_singleop = [{
    "op": "SparseTensorDenseMatMul",
    "input_desc": [
        {"format": "ND", "shape": [NNZ, 2], "type": "int64"},  # indices
        {"format": "ND", "shape": [NNZ], "type": "float"},     # values
        {"format": "ND", "shape": [2], "type": "int64"},       # dense_shape
        {"format": "ND", "shape": [N, 1], "type": "float"},    # x2
    ],
    "output_desc": [{"format": "ND", "shape": [M, 1], "type": "float"}],
    "attr": [
        {"name": "adjoint_a", "type": "bool", "value": False},
        {"name": "adjoint_b", "type": "bool", "value": False},
    ],
}]

json_path = MODEL_DIR / "spmv_singleop.json"
json_path.write_text(json.dumps(spmv_singleop, indent=2, ensure_ascii=False), encoding="utf-8")
print("已写入", json_path)


In [ ]:
subprocess.run([
    "atc", f"--singleop={json_path}", f"--output={MODEL_DIR}",
    f"--soc_version={SOC_VERSION}",
], check=True)


### 在设备上计算

`AclSession` 创建 Context、Stream 和 Event，并在这段代码结束或报错时释放资源。
`input_tensor` 搬入输入，并创建 TensorDesc/DataBuffer；`output_tensor` 准备输出缓冲区。
这些重复操作的完整代码见 [acl_helpers.py](src/acl_helpers.py)。

下面保留算子调用、Event 计时、同步和结果回读。阅读时重点看传入算子的参数，以及同步发生的位置。

输入由 `acl_helpers.py` 将数组转成 bytes，再用 `bytes_to_ptr` 完成同步 H2D。输出先复制到 `malloc_host` 申请的 Host 内存，再用 `ptr_to_bytes` 和 `np.frombuffer` 还原数组。


In [ ]:
with AclSession(MODEL_DIR, DEVICE_ID) as session:
    idx_desc, idx_buffer = session.input_tensor(indices, ACL_INT64)
    val_desc, val_buffer = session.input_tensor(values, ACL_FLOAT)
    shape_desc, shape_buffer = session.input_tensor(dense_shape, ACL_INT64)
    x_desc, x_buffer = session.input_tensor(x2, ACL_FLOAT)

    out_host = np.empty((M, 1), dtype=np.float32)
    out_ptr, out_desc, out_buffer = session.output_tensor(out_host, ACL_FLOAT)
    attr = session.create_attr()
    check(acl.op.set_attr_bool(attr, "adjoint_a", False), "set adjoint_a")
    check(acl.op.set_attr_bool(attr, "adjoint_b", False), "set adjoint_b")

    input_descs = [idx_desc, val_desc, shape_desc, x_desc]
    input_buffers = [idx_buffer, val_buffer, shape_buffer, x_buffer]
    # 重复算子调用，Event 只统计这一段设备执行时间。
    check(acl.rt.record_event(session.start_event, session.stream), "acl.rt.record_event start")
    for _ in range(REPEAT):
        check(acl.op.execute_v2(
            "SparseTensorDenseMatMul",
            input_descs, input_buffers,
            [out_desc], [out_buffer], attr, session.stream,
        ), "acl.op.execute_v2")
    check(acl.rt.record_event(session.end_event, session.stream), "acl.rt.record_event end")
    check(acl.rt.synchronize_stream(session.stream), "acl.rt.synchronize_stream")
    elapsed_ms, ret = acl.rt.event_elapsed_time(session.start_event, session.end_event)
    check(ret, "acl.rt.event_elapsed_time")
    host_ptr, ret = acl.rt.malloc_host(out_host.nbytes)
    check(ret, "acl.rt.malloc_host")
    try:
        check(acl.rt.memcpy(
            host_ptr, out_host.nbytes,
            out_ptr, out_host.nbytes, MEMCPY_D2H,
        ), "acl.rt.memcpy D2H")
        result_bytes = acl.util.ptr_to_bytes(host_ptr, out_host.nbytes)
    finally:
        acl.rt.free_host(host_ptr)
    out_host = np.frombuffer(result_bytes, dtype=out_host.dtype).reshape(out_host.shape)


### 与 NumPy 结果比较

用 COO 坐标逐项累加得到 reference。本例检查最大绝对误差，使用 `atol=1e-3, rtol=0`。reference_ms 用于观察本次参考计算耗时，设备和参考采用不同计时方式，先用于校验结果。


In [ ]:
reference_start = time.perf_counter()
golden = np.zeros(M, dtype=np.float32)
for (row, col), value in zip(indices, values):
    golden[row] += value * x2[col, 0]
reference_ms = (time.perf_counter() - reference_start) * 1000.0
max_abs_error = float(np.max(np.abs(out_host[:, 0] - golden)))
np.testing.assert_allclose(out_host[:, 0], golden, atol=1e-3, rtol=0)

spmv_result = {
    "status": "PASS",
    "actual_backend": f"AICPU/tf_kernel capability path on {SOC_VERSION}/CANN 9.0",
    "aicore_accelerated": False,
    "device_id": int(session.current_device),
    "shape": [M, N],
    "nnz": NNZ,
    "repeat": REPEAT,
    "device_mean_ms": elapsed_ms / REPEAT,
    "reference_ms": reference_ms,
    "max_abs_error": max_abs_error,
    "fallback": 0,
}
print("SPMV_REPORT=" + json.dumps(spmv_result, ensure_ascii=False))


### 课后实践

学生修改：把参数单元格中的 `M、N、NNZ` 改为 `64、128、100`，先运行参数单元格，再从“准备输入”开始向下重新运行。坐标仍须唯一且不越界。记录后端分类、Device ID、平均设备时间和最大误差，并说明为什么 reference 可以直接按 `(row, col, value)` 累加。

输入和单算子描述都会随参数更新；检查本次设备输出与 NumPy 结果是否一致。


完成本节练习后查看：[参考答案](answer/05.03_spmv_answer.md)
